# The Order Of Operations, And Positions

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.9.0

**Author**: Mateusz Jakub Bała

## About

An analysis is a chain, and every operation in it acts on what the operations
before it kept. A condition keeps what passes and leaves out the rest; a count
counts what is left; a position is counted among what is left. So two chains
made of the same operations in another order can ask two different questions
- and both answer right.

This notebook shows where the order matters and why: two ways of asking for
events with two annihilation gammas, the first hit of a track deciding about
the whole track, the second Compton scattering against the second hit, and
why the first example of the analysis concept ends with a count.

## What will you learn from this notebook

- tell apart a count asked before a condition from one asked after it
- put a condition on the hit or the track at a position, and know that it
  decides about the track or the event holding it
- count positions among the hits kept so far
- close a chain with a count when every track of an event has to pass

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.8.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file carried in the repository, described in
[`../data/README.md`](../data/README.md): 500 hits of a scene with a
`PositroniumSource` configured with every channel at once, beside a
back-to-back source. Nothing is written, so no output directory is needed.

In [2]:
import pandas as pd

from opengate_gate_tree import (
    DecayType,
    EventTrackHit,
    EventTrackHitAnalysis,
    EventTrackHitField,
    GammaType,
    GateTree,
    Process,
    SourceType,
    read_tree,
)

Analysis = EventTrackHitAnalysis
Field = EventTrackHitField
data = read_tree(paths.input_example("positronium-source-all-variants.root"), GateTree.HITS)
representation = EventTrackHit(data)
representation

EventTrackHit(events=115, tracks=311, hits=500)

### A Count Counts What Is Left

Two chains of the same three operations:

In [3]:
annihilation_then_two = Analysis().tracks().gamma_type(GammaType.ANNIHILATION).tracks(2)
two_then_annihilation = Analysis().tracks(2).gamma_type(GammaType.ANNIHILATION)

(
    annihilation_then_two.apply(representation).event_count,
    two_then_annihilation.apply(representation).event_count,
)

(46, 35)

The first keeps the annihilation gammas, then the events in which two of them
are left: the para-positronium decays and the direct annihilations, and also
the eleven para-positronium decays with a prompt gamma beside the two. The
second keeps the events with two gammas of any kind first - which leaves out
those eleven - and then their annihilation gammas, which leaves out the
back-to-back events, whose gammas are not annihilation gammas of a positronium
source. 46 and 35: two questions, both answered right.

Closing the second with the same count changes nothing here, since no event of
two gammas holds exactly one annihilation gamma:

In [4]:
two_then_annihilation.tracks(2).apply(representation).event_count

35

### A Position Decides About Its Parent

`hit(1)` stands at the first hit of every track. A condition asked there does
not keep or leave out hits: it keeps or leaves out the whole track. The tracks
whose first interaction was a Compton scattering, with all their hits:

In [5]:
first_compton = Analysis().hit(1).process(Process.COMPTON).apply(representation)
first_compton

EventTrackHit(events=92, tracks=131, hits=320)

In [6]:
first_compton.hits_frame([Field.PROCESS_NAME]).head(6)

,eventIndex,trackIndex,trackNumber,hitNumber,processName
2,0,0,1,1,Compton
3,0,0,1,2,PhotoElectric
0,0,1,2,1,Compton
1,0,1,2,2,PhotoElectric
4,1,2,1,1,Compton
5,1,2,1,2,Compton


Every track kept starts with a Compton scattering, and keeps the hits that
follow it. A track without a hit at the position - a track of one hit, asked
about `hit(2)` - fails the condition and is left out.

`track(1)` does the same one level up: the events whose first track is a prompt
gamma, with all their tracks:

In [7]:
Analysis().track(1).gamma_type(GammaType.PROMPT).apply(representation)

EventTrackHit(events=30, tracks=109, hits=180)

### Positions Are Counted Among What Is Kept

After keeping the Compton scatterings, `hit(2)` is the second Compton
scattering of a track, not its second hit:

In [8]:
second_compton = Analysis().hits().process(Process.COMPTON).hit(2).edep(ge=0.0)
second_hit_compton = Analysis().hit(2).process(Process.COMPTON)

second_compton.apply(representation), second_hit_compton.apply(representation)

(EventTrackHit(events=39, tracks=44, hits=103),
 EventTrackHit(events=39, tracks=44, hits=147))

The same 44 tracks of 39 events answer both here - a gamma that scattered twice
scattered at its first two hits - but not the same hits: the first chain kept
only the Compton scatterings, 103 of them, the second the whole tracks, 147
hits. The `edep(ge=0.0)` after `hit(2)` is a condition every hit passes, there
to make the position decide.

Positions do not nest. `track(1).hit(1)` would read as the first hit of the
first track but would stand at the first hit of every track, so it is refused:

In [9]:
try:
    Analysis().track(1).hit(1)
except ValueError as error:
    print(error)

hit(1) after track(1) would stand at the hits of every track, not of track 1: positions do not nest. Move to tracks() or events() first.


### Closing A Chain With A Count

The first example of the analysis concept asks for the para-positronium decays
whose two annihilation gammas each scattered once and were then absorbed.
Written without its closing count:

In [10]:
open_chain = (
    Analysis()
    .tracks(2)
    .gamma_type(GammaType.ANNIHILATION)
    .decay_type(DecayType.STANDARD)
    .source_type(SourceType.PARA_POSITRONIUM)
    .hits(eq=2)
    .hit(1)
    .process(Process.COMPTON)
    .hit(2)
    .process(Process.PHOTOELECTRIC)
)
kept = open_chain.apply(representation)
kept, pd.Series(kept.track_counts()).value_counts().to_dict()

(EventTrackHit(events=11, tracks=12, hits=24), {1: 10, 2: 1})

Eleven events, but ten of them kept a single gamma: the other one did not
scatter once and get absorbed, and a condition leaves out what fails it rather
than the whole event. A chain does not remember that it started with
`tracks(2)`. Asking for both gammas is asking again, at the end:

In [11]:
open_chain.tracks(2).apply(representation)

EventTrackHit(events=1, tracks=2, hits=4)

One event of this small file. The same chain on a whole simulation of
para-positronium finds many more - the package checks it against the 18
events of the pPs fixture of its tests.

## Summary

Every operation acts on what the ones before it kept: a count counts what is
left, a position is counted among what is left, and a condition at a position
decides about the track or event holding it. Changing the order changes the
question, and a chain that needs every track of an event to pass ends with a
count that says so.

## Bibliography

- [Analysis Of Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/analysis.html)
  — where an analysis stands, conditions as filters, and the order of operations
- [Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/event-track-hit.html)
  — the representation an analysis is performed on, and the order it counts in
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — what `sourceType`, `decayType` and `gammaType` mean

## Changelog

- v1: written.